# DocBank → YOLO → OCR pipeline — **Colab edition**

Same `docbank_pipeline` package as the local notebook, with three Colab-specific tweaks:

1. **Mount Google Drive** so artefacts persist across Colab sessions.
2. **Clone the project repo** (or upload it) so the package is importable.
3. **Use Colab's GPU** (Runtime → Change runtime type → T4 GPU).

Pipeline stages are unchanged from the local notebook:
1. Configure paths · 2. Install deps · 3. Download + extract DocBank · 4. Convert COCO → YOLO · 5. Train · 6. Validate · 7. Inference + OCR · 8. Smoke test.

## 0. Mount Google Drive

Drive holds the dataset, weights and results so a Colab session timeout doesn't lose anything.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clone the project into Colab

The project source lives at **https://github.com/Coffeeandtea2/docbank-pipeline**.

Colab gives you a fresh container every session, so this clone runs once at the start of each session. It only pulls source code (~200 KB) — the dataset, weights and outputs all live in your Drive (configured in step 3).

> If the repo is private, replace the URL with `https://<token>@github.com/Coffeeandtea2/docbank-pipeline.git` where `<token>` is a Personal Access Token from <https://github.com/settings/tokens> (scope: `repo`).

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/sadsa')

# `git clone` is a no-op if the directory is already populated (re-running
# the cell after a previous clone is safe). Pull latest changes either way.
if not PROJECT_DIR.exists() or not (PROJECT_DIR / 'docbank_pipeline').is_dir():
    !git clone https://github.com/Coffeeandtea2/docbank-pipeline.git /content/sadsa
else:
    !git -C /content/sadsa pull --ff-only

os.chdir(PROJECT_DIR)
print('Project at:', PROJECT_DIR)
print('Contains:', sorted(p.name for p in PROJECT_DIR.iterdir()))

## 2. Install dependencies

Colab already has torch + numpy + pillow + opencv. We add the project-specific bits.

In [ ]:
%pip install -q huggingface_hub ultralytics tqdm
# OCR / formula recognition (heavy — comment out if you only need detection):
%pip install -q 'paddlepaddle==2.6.2' 'paddleocr<3' pix2tex
# Web app (optional in Colab):
# %pip install -q flask waitress pymupdf

## 3. Configure paths (Drive-backed)

Putting `DATA_ROOT` under Drive means the next Colab session can pick up exactly where this one left off — every stage's `.done` marker survives.

In [ ]:
from pathlib import Path
import sys, os

PROJECT_DIR = Path('/content/sadsa')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# --- TWEAK ME -----------------------------------------------------------
DATA_ROOT       = Path('/content/drive/MyDrive/DocBank')   # persists across sessions
DATASET_PARTS   = 1                                         # 1..10 of the image archive
MAX_PAGES       = 1000                                      # cap per split for fast runs
NUM_WORKERS     = 2                                         # Colab boxes are small
TRAIN_VAL_SPLIT = 0.9
# Optional: paste your HuggingFace token to dodge anonymous rate limits.
# os.environ['HF_TOKEN'] = 'hf_...'
# ------------------------------------------------------------------------

from docbank_pipeline import PipelineConfig, setup_logging
log = setup_logging()

cfg = PipelineConfig(
    data_root=DATA_ROOT,
    dataset_parts=DATASET_PARTS,
    max_pages=MAX_PAGES,
    num_workers=NUM_WORKERS,
    train_val_split=TRAIN_VAL_SPLIT,
)
cfg.ensure_dirs()
print(cfg.describe())

### GPU sanity check

If `Runtime → Change runtime type → T4 GPU` is set, this should print `cuda`. If it prints `cpu`, training will work but be ~10× slower.

In [ ]:
from docbank_pipeline import detect_device
print('Detected device:', detect_device())

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## 4. Download + extract DocBank

Resumable: huggingface_hub skips files already in the Drive cache, and `extract_archives` writes a `.extract.done` marker.

Colab has 7-Zip preinstalled (`apt list --installed | grep p7zip`), so partial-archive extraction works out of the box.

In [ ]:
# 7-Zip on Colab (no-op if already installed):
!apt -qq install p7zip-full > /dev/null

from docbank_pipeline import download_docbank_parts, verify_downloads, extract_archives

download_docbank_parts(cfg)
verify_downloads(cfg)
extract_archives(cfg)

## 5. Convert COCO → YOLO

Builds a single `{filename → path}` index once, then each lookup is O(1).

In [ ]:
from docbank_pipeline import split_dataset, dataset_statistics

split_dataset(cfg)
for split, stats in dataset_statistics(cfg).items():
    print(f'{split:>5}: {stats}')

## 6. Train YOLOv8 on Colab GPU

With a free T4 (15 GB VRAM) you can comfortably go to `imgsz=1024, batch=16, model='yolov8s.pt'`. Adjust below if you hit OOM.

In [ ]:
from docbank_pipeline import train_yolo

results = train_yolo(
    cfg,
    model='yolov8n.pt',   # try 'yolov8s.pt' or 'yolov8m.pt' on T4
    epochs=50,            # Colab GPU finishes 50 epochs in ~30 min
    imgsz=640,
    batch=16,             # raise to 32 on T4 if VRAM allows
    run_name='yolov8n_docbank_colab',
)
print('Best weights:', results.save_dir / 'weights' / 'best.pt')

## 7. Validate

In [ ]:
from docbank_pipeline import validate_yolo

metrics = validate_yolo(cfg, split='val')
print(f'mAP50    = {metrics.box.map50:.3f}')
print(f'mAP50-95 = {metrics.box.map:.3f}')

## 8. Inference + OCR + LaTeX-OCR

On the GPU, PaddleOCR and pix2tex are ~10× faster, so the 1000-page run that took 45 min on CPU should finish in roughly 5 min.

In [ ]:
from docbank_pipeline import process_folder

demo_source = cfg.yolo_dataset_dir / 'images' / 'val'
out_json    = cfg.output_dir / 'results.json'

results = process_folder(
    cfg,
    demo_source,
    conf=0.25,
    do_text=True,
    do_formula=True,
    output_json=out_json,
    max_pages=1000,
    min_ocr_conf=0.30,
    min_ocr_area=600,
    min_formula_area=2000,
)
print(f'{len(results)} detection(s) written to {out_json}')

### Inspect a few JSON records

In [ ]:
import json
with open(cfg.output_dir / 'results.json', encoding='utf-8') as f:
    payload = json.load(f)

print(f'pages: {len(payload["pages"])}')
for page in payload['pages'][:1]:
    print('image:', page['image'])
    for det in page['detections'][:5]:
        snippet = (det.get('recognized') or '')[:60].replace(chr(10), ' / ')
        print(f"  {det['class_name']:<7} conf={det['confidence']:.2f} bbox={det['bbox']} text={snippet!r}")

## 9. Save artefacts back to Drive

If you set `DATA_ROOT` under Drive in step 3, this is automatic — every save already lands in your Drive. The cells below are just for explicit copies (e.g. of the trained model out to a known path).

In [ ]:
import shutil
from pathlib import Path

DRIVE_BACKUP = Path('/content/drive/MyDrive/docbank_artifacts')
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

best_pt = next(cfg.runs_dir.glob('**/weights/best.pt'), None)
if best_pt:
    shutil.copy2(best_pt, DRIVE_BACKUP / 'best.pt')
    print(f'Copied weights -> {DRIVE_BACKUP / "best.pt"}')

results_p = cfg.output_dir / 'results.json'
if results_p.is_file():
    shutil.copy2(results_p, DRIVE_BACKUP / 'results.json')
    print(f'Copied results.json -> {DRIVE_BACKUP / "results.json"}')

## 10. Smoke-test (5-20 pages)

End-to-end run on a tiny subset, useful when you've just (re-)mounted Drive and want to confirm the whole stack still works.

In [ ]:
from docbank_pipeline.cli import main as cli_main

cli_main(['smoketest', '--n', '10', '--epochs', '2',
          '--data-root', str(cfg.data_root)])

## Cheatsheet — CLI equivalents

```bash
!python -m docbank_pipeline info
!python -m docbank_pipeline download --dataset-parts 1
!python -m docbank_pipeline extract
!python -m docbank_pipeline convert --max-pages 1000
!python -m docbank_pipeline train --epochs 50 --batch 16
!python -m docbank_pipeline val
!python -m docbank_pipeline infer \
    --source DocBank/yolo_dataset/images/val \
    --output DocBank/outputs/results.json \
    --max-pages 1000 \
    --min-ocr-conf 0.30 \
    --min-ocr-area 600 \
    --min-formula-area 2000
```

All paths flow from `--data-root` / the `DATA_ROOT` env var, so you can point them at any Drive folder without code changes.